# Notebook 08b: Vehicle ReID — TransReID Training (v8+, Triplet)
**Multi-Camera Tracking System — Kaggle Training Pipeline**

Train SOTA vehicle re-identification on VeRi-776 using TransReID ViT-Base with CLIP pretraining.

## Model
| Model | Architecture | Dim | Target mAP | Target R1 |
|-------|-------------|-----|-------------|-------------|
| **TransReID** | ViT-Base/16 (CLIP) + SIE + JPM | 768 | >82% | >97% |

## Key Features (v8+)
- All v6 fixes: norm_pre, SIE all tokens, LLRD(0.75), OpenAI CLIP
- v6 proven hyperparameters: backbone_lr=3.5e-4, head_lr=3.5e-3
- **Triplet loss** (margin=0.3) — proven in v8 (80.4% mAP)
- **Center loss** activated after epoch 30 (delayed start for stability)
- **Label smoothing ε=0.15** (up from v8's 0.1 — helps with similar vehicles)
- **180 epochs** (v8 was 140; longer cosine tail for fine-tuning)
- **Random Erasing** (p=0.5, random fill — proven in TransReID paper)
- v6 augmentation base (NO AutoAugment — v7 lesson)

**v8 → v8+ changes:**
| Change | v8 | v8+ | Rationale |
|--------|----|----|-----------|
| Label smooth ε | 0.1 | 0.15 | VeRi has many similar vehicles; stronger smoothing helps |
| Epochs | 140 | 180 | Longer cosine tail benefits CLIP fine-tuning |

**Runtime**: GPU T4 x2 (32GB) | **Duration**: ~3.5h
**Parallel run with 08 v9 (Circle Loss) — head-to-head comparison.**

In [ ]:
!pip install -q timm matplotlib pandas

## 1. Setup

In [ ]:
import os
import sys
import json
import time
import re
import glob
import shutil
import os.path as osp
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim import Adam, SGD
from torch.optim.lr_scheduler import LambdaLR
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
NUM_GPUS = max(torch.cuda.device_count(), 1)
print(f"GPUs available: {NUM_GPUS}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1024**3:.1f} GB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working/vehicle_reid_sota")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/kaggle/working/exported_models")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nDevice: {DEVICE} | GPUs: {NUM_GPUS}")

## 2. Dataset Preparation (VeRi-776)

In [ ]:
# â”€â”€ Locate VeRi-776 from Kaggle input â”€â”€
DATA_ROOT = Path("/kaggle/working/data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

possible_roots = [
    Path("/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset"),
    Path("/kaggle/input/veri-vehicle-re-identification-dataset"),
]

input_dir = None
for root in possible_roots:
    if root.exists():
        input_dir = root
        break

if input_dir is None:
    import subprocess
    result = subprocess.run(
        ["find", "/kaggle/input", "-maxdepth", "5", "-type", "d", "-name", "image_train"],
        capture_output=True, text=True, timeout=30
    )
    print(f"Search:\n{result.stdout}")
    if result.stdout.strip():
        found = Path(result.stdout.strip().split("\n")[0]).parent
        input_dir = found
    else:
        raise RuntimeError("VeRi-776 not found. Attach 'abhyudaya12/veri-vehicle-re-identification-dataset'.")

# Find actual data (may be nested)
veri_data_dir = None
for p in input_dir.rglob("image_train"):
    if p.is_dir():
        veri_data_dir = p.parent
        break

if veri_data_dir is None:
    raise RuntimeError(f"No 'image_train' in {input_dir}")

print(f"VeRi data at: {veri_data_dir}")

# Symlink
symlink = DATA_ROOT / "veri"
if symlink.exists() or symlink.is_symlink():
    if symlink.is_symlink():
        symlink.unlink()
    else:
        shutil.rmtree(symlink)
symlink.symlink_to(veri_data_dir)

for subdir in ["image_train", "image_query", "image_test"]:
    d = symlink / subdir
    n = len(list(d.glob("*.jpg"))) if d.exists() else 0
    print(f"  {subdir}: {n} images")

In [ ]:
# â”€â”€ Parse VeRi-776 â”€â”€
def parse_veri776(root):
    """Parse VeRi-776. Format: XXXX_cYYY_ZZZZZZZZ_T.jpg"""
    pat = re.compile(r'^(\d+)_c(\d+)')
    train, query, gallery = [], [], []
    for split_name, split_list in [
        ("image_train", train), ("image_query", query), ("image_test", gallery),
    ]:
        split_dir = os.path.join(root, split_name)
        for fname in sorted(os.listdir(split_dir)):
            if not fname.endswith(".jpg"):
                continue
            m = pat.match(fname)
            if m:
                pid = int(m.group(1))
                cam = int(m.group(2)) - 1
                split_list.append((os.path.join(split_dir, fname), pid, cam))

    # Re-label train PIDs
    train_pids = sorted(set(pid for _, pid, _ in train))
    pid2label = {pid: i for i, pid in enumerate(train_pids)}
    train = [(p, pid2label[pid], c) for p, pid, c in train]
    return train, query, gallery

train_data, query_data, gallery_data = parse_veri776(str(symlink))
num_classes = len(set(pid for _, pid, _ in train_data))
num_cameras = len(set(cam for _, _, cam in train_data))

print(f"Train: {len(train_data)} images, {num_classes} IDs, {num_cameras} cameras")
print(f"Query: {len(query_data)} images")
print(f"Gallery: {len(gallery_data)} images")

## 3. Data Loading + Losses

In [ ]:
# -- CLIP normalization constants --
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]
H, W = 224, 224

# -- Transforms (v6-v9: proven augmentation, NO AutoAugment) --
# Random Erasing (p=0.5, random fill) — proven in TransReID paper.
# Applied AFTER normalize on tensor values, with random noise fill
# (not zeros — random fill is more robust, prevents the model from
# learning to detect erased regions by their constant value).
train_tf = T.Compose([
    T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((H, W)),
    T.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.1, hue=0),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
    T.RandomErasing(p=0.5, value="random"),
])

test_tf = T.Compose([
    T.Resize((H, W), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
])

print(f"Using CLIP normalization: mean={CLIP_MEAN}")
print("Augmentation: HFlip + Pad+Crop + ColorJitter + RandomErasing(p=0.5,random) (v6-v9 proven)")

# -- Dataset + PK Sampler --
class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        path, pid, cam = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, pid, cam, path

class PKSampler(Sampler):
    def __init__(self, data_source, p=16, k=4):
        self.p, self.k = p, k
        self.pid_to_idx = defaultdict(list)
        for i, (_, pid, _) in enumerate(data_source):
            self.pid_to_idx[pid].append(i)
        self.pids = list(self.pid_to_idx.keys())
        self.batch_size = p * k
    def __iter__(self):
        np.random.shuffle(self.pids)
        batch = []
        for pid in self.pids:
            idxs = self.pid_to_idx[pid]
            sel = np.random.choice(idxs, self.k, replace=len(idxs) < self.k).tolist()
            batch.extend(sel)
            if len(batch) >= self.batch_size:
                yield from batch[:self.batch_size]
                batch = batch[self.batch_size:]
        if batch:
            yield from batch
    def __len__(self):
        return len(self.pids) * self.k

# -- Build loaders --
BATCH = 48 * NUM_GPUS  # 96 on 2xT4
P_IDS = 12 * NUM_GPUS  # 24 IDs per batch

train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

sampler = PKSampler(train_data, p=P_IDS, k=4)
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                          num_workers=4, pin_memory=True, drop_last=True)

query_loader = DataLoader(query_ds, batch_size=128, num_workers=4, pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=128, num_workers=4, pin_memory=True)
print(f"Train: {len(train_loader)} batches (batch={BATCH}) | Query: {len(query_loader)} | Gallery: {len(gallery_loader)}")

In [ ]:
# ── Losses (numerically stable for fp16 + DataParallel) ──
class CrossEntropyLabelSmooth(nn.Module):
    """Label-smooth CE using F.log_softmax in fp32 for numerical stability."""
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon

    def forward(self, inputs, targets):
        # Force fp32 for log_softmax to prevent overflow
        log_probs = F.log_softmax(inputs.float(), dim=1)
        with torch.no_grad():
            oh = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
            smooth = (1 - self.epsilon) * oh + self.epsilon / self.num_classes
        loss = (-smooth * log_probs).sum(dim=1).mean()
        return loss


class TripletLossHardMining(nn.Module):
    """Triplet loss with batch hard mining — fp32 computation.
    
    For each anchor, find hardest positive (max dist) and hardest negative
    (min dist) within the batch. Standard in TransReID / BoT.
    """
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, pids):
        feats = F.normalize(feats.float(), p=2, dim=1)
        n = feats.size(0)
        dist = torch.cdist(feats, feats, p=2)  # (n, n) Euclidean
        
        mask_pos = pids.unsqueeze(0).eq(pids.unsqueeze(1))
        mask_neg = ~mask_pos
        
        # For each anchor: hardest positive (largest distance)
        dist_pos = dist.clone()
        dist_pos[~mask_pos] = 0
        hardest_pos, _ = dist_pos.max(dim=1)
        
        # For each anchor: hardest negative (smallest distance)
        dist_neg = dist.clone()
        dist_neg[~mask_neg] = float('inf')
        hardest_neg, _ = dist_neg.min(dim=1)
        
        y = torch.ones(n, device=feats.device)
        return self.ranking_loss(hardest_neg, hardest_pos, y)


class CenterLoss(nn.Module):
    """Center loss (Wen et al., ECCV 2016) — fp32 computation."""
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, feats, labels):
        feats = feats.float()
        batch_size = feats.size(0)
        centers_batch = self.centers[labels]  # (B, D)
        diff = feats - centers_batch
        loss = (diff * diff).sum(1).mean()
        return loss


print("Losses defined: CE+LS(ε=0.15), TripletLoss(m=0.3), CenterLoss (fp32-stable)")

## 4. Evaluation Functions

In [ ]:
@torch.no_grad()
def extract_features(model, loader, device="cuda", flip=True, pass_cams=False):
    """Extract L2-normalized features. pass_cams=True for TransReID SIE."""
    model.eval()
    feats, pids, cams = [], [], []
    for imgs, pid, cam, _ in loader:
        imgs = imgs.to(device)
        kwargs = {"cam_ids": cam.to(device).long()} if pass_cams else {}
        f = model(imgs, **kwargs)
        if isinstance(f, (tuple, list)):
            f = f[-1]
        if flip:
            ff = model(torch.flip(imgs, [3]), **kwargs)
            if isinstance(ff, (tuple, list)):
                ff = ff[-1]
            f = (f + ff) / 2
        f = F.normalize(f, p=2, dim=1)
        feats.append(f.cpu().numpy())
        pids.append(pid.numpy())
        cams.append(cam.numpy())
    return np.concatenate(feats), np.concatenate(pids), np.concatenate(cams)

def eval_market1501(distmat, q_pids, g_pids, q_cams, g_cams, max_rank=50):
    nq = distmat.shape[0]
    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, None]).astype(np.int32)
    all_cmc, all_AP = [], []
    for qi in range(nq):
        order = indices[qi]
        remove = (g_pids[order] == q_pids[qi]) & (g_cams[order] == q_cams[qi])
        raw_cmc = matches[qi][~remove]
        if raw_cmc.sum() == 0:
            continue
        cmc = raw_cmc.cumsum(); cmc[cmc > 1] = 1
        all_cmc.append(cmc[:max_rank])
        n_rel = raw_cmc.sum()
        tmp = raw_cmc.cumsum()
        prec = tmp / (np.arange(len(tmp)) + 1.0)
        all_AP.append((prec * raw_cmc).sum() / n_rel)
    return float(np.mean(all_AP)), np.array(all_cmc).mean(0)

def compute_reranking(qf, gf, k1=20, k2=6, lam=0.3):
    all_f = np.concatenate([qf, gf], axis=0)
    N, nq = all_f.shape[0], qf.shape[0]
    od = np.clip(2.0 - 2.0 * (all_f @ all_f.T), 0, None)
    ir = np.argsort(od, axis=1)
    V = np.zeros((N, N), dtype=np.float32)
    for i in range(N):
        fwd = ir[i, :k1+1]
        kr = [c for c in fwd if i in ir[c, :k1+1]]
        kr = np.array(kr); kr_exp = kr.copy()
        for c in kr:
            cf = ir[c, :int(round(k1/2))+1]
            ckr = [cc for cc in cf if c in ir[cc, :int(round(k1/2))+1]]
            if len(ckr) > 2/3 * len(cf):
                kr_exp = np.union1d(kr_exp, ckr)
        w = np.exp(-od[i, kr_exp])
        V[i, kr_exp] = w / (w.sum() + 1e-12)
    if k2 > 0:
        Vqe = np.zeros_like(V)
        for i in range(N):
            Vqe[i] = V[ir[i, :k2+1]].mean(0)
        V = Vqe
    jac = np.zeros((nq, N-nq), dtype=np.float32)
    for i in range(nq):
        mn = np.minimum(V[i], V[nq:]); mx = np.maximum(V[i], V[nq:])
        jac[i] = 1 - mn.sum(1) / (mx.sum(1) + 1e-12)
    return jac * (1 - lam) + od[:nq, nq:] * lam

def full_eval(model, ql, gl, device="cuda", rerank=True, pass_cams=False):
    qf, qp, qc = extract_features(model, ql, device, pass_cams=pass_cams)
    gf, gp, gc = extract_features(model, gl, device, pass_cams=pass_cams)
    dist = 1.0 - qf @ gf.T
    mAP, cmc = eval_market1501(dist, qp, gp, qc, gc)
    mAP_rr, cmc_rr = None, None
    if rerank:
        dist_rr = compute_reranking(qf, gf)
        mAP_rr, cmc_rr = eval_market1501(dist_rr, qp, gp, qc, gc)
    return mAP, cmc, mAP_rr, cmc_rr

print("Evaluation ready (with SIE-aware feature extraction)")

## 5. TransReID Model

Camera-aware SIE is especially important for vehicles (same car looks very
different across 20 cameras with different viewpoints/lighting).

**v6 Critical Fixes (carried into v8+):**
- **norm_pre**: CLIP ViTs have a pre-LayerNorm before transformer blocks.
- **SIE on ALL tokens**: Original TransReID broadcasts camera embedding to all patch tokens.
- **LLRD**: Layer-wise LR Decay protects shallow CLIP features.
- **Pure OpenAI CLIP**: Uses `vit_base_patch16_clip_224.openai`.

**v8+ Training (conservative, proven base):**
- **Triplet Loss** (margin=0.3): Same as v8. Proven, reliable metric learning.
- **Label smoothing ε=0.15**: Up from 0.1.
- **180 epochs**: Extended from 140.

In [ ]:
import timm

# ── Select CLIP ViT-Base backbone ──
def find_clip_vit_base():
    """Find best CLIP ViT-Base/16 in timm — prefer pure OpenAI CLIP."""
    try:
        tags = timm.list_pretrained('vit_base_patch16_clip*224*')
    except Exception:
        tags = []
    print(f"Available CLIP pretrained tags: {tags}")

    # Prefer pure OpenAI CLIP (what CLIP-ReID paper uses for 82.1% mAP)
    for t in sorted(tags):
        if 'openai' in t and 'ft' not in t:
            return t
    # Then OpenAI fine-tuned on ImageNet
    for t in sorted(tags):
        if 'openai' in t:
            return t
    # Then any CLIP variant
    for t in sorted(tags):
        if 'laion' in t:
            return t
    if tags:
        return sorted(tags)[0]

    # Fallback for older timm without pretrained tags
    clip_models = timm.list_models("*vit_base*patch16*clip*224*")
    if clip_models:
        return sorted(clip_models)[0]
    clip_models = timm.list_models("*vit_base*patch16*clip*")
    if clip_models:
        return sorted(clip_models)[0]
    raise RuntimeError("No CLIP ViT-Base found in timm! Update timm.")

VIT_MODEL = find_clip_vit_base()
print(f"Selected backbone: {VIT_MODEL}")


class TransReID(nn.Module):
    """TransReID: ViT-Base + SIE + JPM (He et al., ICCV 2021).

    v6 CRITICAL FIX: Includes timm's norm_pre for CLIP compatibility.
    CLIP ViTs use pre-LayerNorm that standard ViTs lack — skipping it
    completely destroys pretrained features (root cause of v3-v5 failure).
    """
    def __init__(self, num_classes, num_cameras=0, embed_dim=768,
                 vit_model="vit_base_patch16_clip_224", pretrained=True,
                 sie_camera=True, jpm=True):
        super().__init__()
        self.sie_camera = sie_camera and num_cameras > 0
        self.jpm = jpm
        self.vit = timm.create_model(vit_model, pretrained=pretrained, num_classes=0)
        self.vit_dim = self.vit.embed_dim  # 768 for ViT-Base
        cfg = getattr(self.vit, 'pretrained_cfg', {})
        print(f"  Pretrained: {cfg.get('hf_hub_id', cfg.get('url', 'unknown'))[:80]}")

        # Detect architecture features
        has_norm_pre = hasattr(self.vit, 'norm_pre') and not isinstance(
            self.vit.norm_pre, nn.Identity)
        print(f"  norm_pre: {type(self.vit.norm_pre).__name__} (active={has_norm_pre})")
        self.num_blocks = len(self.vit.blocks)
        print(f"  Transformer blocks: {self.num_blocks}")

        # SIE: camera embedding broadcast to ALL tokens (original TransReID)
        if self.sie_camera:
            self.sie_embed = nn.Parameter(torch.zeros(num_cameras, 1, self.vit_dim))
            nn.init.trunc_normal_(self.sie_embed, std=0.02)
            print(f"  SIE: {num_cameras} cameras, broadcast to all tokens")

        # BNNeck
        self.bn = nn.BatchNorm1d(self.vit_dim)
        self.bn.bias.requires_grad_(False)

        # Projection + classifier
        self.proj = nn.Linear(self.vit_dim, embed_dim, bias=False) if embed_dim != self.vit_dim else nn.Identity()
        self.cls_head = nn.Linear(embed_dim, num_classes, bias=False)
        if isinstance(self.proj, nn.Linear):
            nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
        nn.init.normal_(self.cls_head.weight, std=0.001)

        # JPM: jigsaw patch module for part-level features
        if self.jpm:
            self.bn_jpm = nn.BatchNorm1d(self.vit_dim)
            self.bn_jpm.bias.requires_grad_(False)
            self.jpm_cls = nn.Linear(self.vit_dim, num_classes, bias=False)
            nn.init.normal_(self.jpm_cls.weight, std=0.001)

        print(f"TransReID: {vit_model}, dim={self.vit_dim}, embed={embed_dim}, "
              f"SIE={self.sie_camera}({num_cameras}), JPM={jpm}, blocks={self.num_blocks}")

    def forward(self, x, cam_ids=None):
        B = x.shape[0]
        # 1. Patch embedding
        x = self.vit.patch_embed(x)

        # 2. CLS token + positional embedding + pos_drop (use timm's method)
        if hasattr(self.vit, '_pos_embed'):
            x = self.vit._pos_embed(x)
        else:
            cls_tok = self.vit.cls_token.expand(B, -1, -1)
            x = torch.cat([cls_tok, x], dim=1) + self.vit.pos_embed
            if hasattr(self.vit, 'pos_drop'):
                x = self.vit.pos_drop(x)

        # 3. SIE: camera embedding broadcast to ALL tokens
        if self.sie_camera and cam_ids is not None:
            x = x + self.sie_embed[cam_ids]  # (B,1,D) broadcasts to (B,N+1,D)

        # 4. Patch drop (Identity for most models)
        if hasattr(self.vit, 'patch_drop'):
            x = self.vit.patch_drop(x)

        # 5. CRITICAL: Pre-normalization (CLIP uses LayerNorm here!)
        #    v3-v5 SKIPPED this — root cause of all CLIP training failures.
        #    Standard ViTs have Identity here, so this only affects CLIP.
        if hasattr(self.vit, 'norm_pre'):
            x = self.vit.norm_pre(x)

        # 6. Transformer blocks
        for blk in self.vit.blocks:
            x = blk(x)
        x = self.vit.norm(x)

        # Global feature (CLS token)
        g = x[:, 0]
        bn = self.bn(g)
        proj = self.proj(bn)

        if self.training:
            cls = self.cls_head(proj)
            if self.jpm:
                patches = x[:, 1:]
                idx = torch.randperm(patches.size(1), device=x.device)
                s = patches[:, idx]
                mid = s.size(1) // 2
                jf = (s[:, :mid].mean(1) + s[:, mid:].mean(1)) / 2
                return cls, proj, self.jpm_cls(self.bn_jpm(jf))
            return cls, proj
        return F.normalize(proj, p=2, dim=1)

    def get_llrd_param_groups(self, backbone_lr, head_lr, decay=0.75):
        """Layer-wise LR decay: shallow layers get exponentially smaller LR.

        Essential for CLIP fine-tuning — preserves pretrained representations
        in early layers while deeper layers adapt more aggressively.

        With decay=0.75 and 12 blocks:
          Block 11 (deepest): backbone_lr * 1.0
          Block 0  (shallowest): backbone_lr * 0.75^11 ≈ backbone_lr * 0.042
          Embeddings: backbone_lr * 0.75^12 ≈ backbone_lr * 0.032
        """
        groups = {}

        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue

            if name.startswith('vit.'):
                if 'blocks.' in name:
                    block_idx = int(name.split('blocks.')[1].split('.')[0])
                    depth = block_idx + 1  # 1-indexed
                elif any(k in name for k in ['patch_embed', 'cls_token', 'pos_embed', 'norm_pre']):
                    depth = 0  # Embedding layer (lowest LR)
                else:
                    depth = self.num_blocks + 1  # Final norm etc

                scale = decay ** (self.num_blocks + 1 - depth)
                lr = backbone_lr * scale
                gk = f"bb_d{depth}"
            else:
                lr = head_lr
                gk = "head"

            if gk not in groups:
                groups[gk] = {"params": [], "lr": lr}
            groups[gk]["params"].append(param)

        result = sorted(groups.values(), key=lambda x: x["lr"])
        # Print summary
        for g in result:
            n = sum(p.numel() for p in g["params"])
            print(f"  lr={g['lr']:.2e} | {n:>12,} params")

        return result


model = TransReID(
    num_classes=num_classes, num_cameras=num_cameras,
    embed_dim=768, vit_model=VIT_MODEL, sie_camera=True, jpm=True,
).to(DEVICE)
if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"  Wrapped in DataParallel ({NUM_GPUS} GPUs)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# -- Training TransReID (CLIP ViT-Base) on VeRi-776 --

# v9: Label smoothing ε=0.15 (up from 0.1 — vehicles are harder to distinguish)
ce_loss = CrossEntropyLabelSmooth(num_classes, 0.15).to(DEVICE)

# v8+: Triplet loss with hard mining (proven in v8: 80.4% mAP)
tri_loss = TripletLossHardMining(margin=0.3).to(DEVICE)

# v8→v9: Center loss -- delayed start at epoch 30 for stability
ctr_loss = CenterLoss(num_classes, 768).to(DEVICE)
CENTER_WEIGHT = 5e-4
CENTER_START = 30  # Only activate AFTER classifier has converged somewhat

raw_model = model.module if hasattr(model, 'module') else model

# v6/v8/v9: Proven CLIP fine-tuning hyperparameters (unchanged)
backbone_lr = 3.5e-4
head_lr = 3.5e-3
wd = 5e-4
llrd_factor = 0.75  # v6 proven value

print(f"LLRD config: decay={llrd_factor}")
print(f"  Deepest backbone layer lr: {backbone_lr:.2e}")
print(f"  Shallowest (embed) layer lr: {backbone_lr * llrd_factor**(raw_model.num_blocks+1):.2e}")
print(f"  Metric loss: TripletLoss(m=0.3)")
print(f"  Center loss weight: {CENTER_WEIGHT} (starts at epoch {CENTER_START})")
print(f"  Label smoothing: ε=0.15")

param_groups = raw_model.get_llrd_param_groups(backbone_lr, head_lr, decay=llrd_factor)
optimizer = torch.optim.AdamW(param_groups, weight_decay=wd)

# v8/v9: Separate center loss optimizer (SGD, lr=0.5, no weight decay -- standard)
center_optimizer = torch.optim.SGD(ctr_loss.parameters(), lr=0.5)

# Store base LRs for warmup
base_lrs = [pg["lr"] for pg in optimizer.param_groups]

n_bb = sum(p.numel() for n, p in raw_model.named_parameters() if "vit" in n)
n_hd = sum(p.numel() for n, p in raw_model.named_parameters() if "vit" not in n)
print(f"Backbone params: {n_bb:,} (max_lr={backbone_lr})")
print(f"Head params:     {n_hd:,} (lr={head_lr})")

# v9: 180 epochs (up from 140 — longer cosine tail for CLIP fine-tuning)
EPOCHS = 180
WARMUP = 10

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP)
scaler = torch.amp.GradScaler("cuda")

history = {"loss": [], "mAP": [], "R1": [], "mAP_rr": [], "R1_rr": []}
best_mAP = 0.0

print("=" * 70)
print(f"  Losses: CE(ε=0.15) + Triplet(0.3) + Center(5e-4, delayed@ep{CENTER_START})")
print(f"  v8→v8+: same Triplet, ε 0.1→0.15, epochs 140→180")
print(f"  LLRD factor={llrd_factor}, warmup={WARMUP}")
print("=" * 70)

t0 = time.time()
for epoch in range(EPOCHS):
    model.train()
    rl, nb = 0.0, 0
    use_center = (epoch >= CENTER_START)  # v8/v9: delayed center loss

    # Warmup: linearly scale all LRs from 0 to base
    if epoch < WARMUP:
        wf = (epoch + 1) / WARMUP
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i] * wf
    elif epoch == WARMUP:
        for i, pg in enumerate(optimizer.param_groups):
            pg["lr"] = base_lrs[i]

    for imgs, pids, cams, _ in train_loader:
        imgs, pids, cams = imgs.to(DEVICE), pids.to(DEVICE).long(), cams.to(DEVICE).long()
        optimizer.zero_grad()
        if use_center:
            center_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            out = model(imgs, cam_ids=cams)
            if len(out) == 3:
                c, f, jc = out
                # v8+: Triplet loss (proven in v8)
                loss = ce_loss(c, pids) + tri_loss(f, pids) + 0.5 * ce_loss(jc, pids)
            else:
                c, f = out
                loss = ce_loss(c, pids) + tri_loss(f, pids)

        if use_center:
            # Center loss in fp32 (outside autocast for numerical stability)
            center_l = ctr_loss(f.float(), pids)
            total_loss = loss + CENTER_WEIGHT * center_l
            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)
            scaler.unscale_(center_optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.step(center_optimizer)
        else:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            scaler.step(optimizer)

        scaler.update()
        rl += loss.item() if not torch.isnan(loss) else 0.0
        nb += 1

    if epoch >= WARMUP:
        scheduler.step()
    history["loss"].append(rl / max(nb, 1))

    if (epoch + 1) % 10 == 0:
        hd_lr = optimizer.param_groups[-1]["lr"]
        top_bb_lr = max(pg["lr"] for pg in optimizer.param_groups[:-1])
        ctr_tag = " [+center]" if use_center else ""
        print(f"Epoch {epoch+1:3d} | Loss {rl/max(nb,1):.4f} | "
              f"top_bb_lr={top_bb_lr:.2e} head_lr={hd_lr:.2e}{ctr_tag}")

    # Evaluate every 20 epochs (and final)
    if (epoch + 1) % 20 == 0 or epoch == EPOCHS - 1:
        mAP, cmc, mAP_rr, cmc_rr = full_eval(model, query_loader, gallery_loader,
                                               DEVICE, pass_cams=True)
        history["mAP"].append(mAP); history["R1"].append(cmc[0])
        history["mAP_rr"].append(mAP_rr or 0)
        history["R1_rr"].append(cmc_rr[0] if cmc_rr is not None else 0)
        is_best = mAP > best_mAP
        if is_best:
            best_mAP = mAP
            _state = (model.module if hasattr(model, 'module') else model).state_dict()
            torch.save(_state, OUTPUT_DIR / "transreid_veri_best.pth")
        tag = " ★" if is_best else ""
        print(f"  → mAP: {mAP:.4f}, R1: {cmc[0]:.4f}")
        if mAP_rr: print(f"  → mAP(RR): {mAP_rr:.4f}, R1(RR): {cmc_rr[0]:.4f}{tag}")

elapsed = time.time() - t0
print(f"\nTransReID (CLIP) v8+ done in {elapsed/3600:.1f}h | Best mAP: {best_mAP:.4f}")

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("TransReID (CLIP) v8+ — VeRi-776", fontsize=14, fontweight="bold")

ax = axes[0]
ax.plot(history["loss"], "b-", alpha=0.7)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("Training Loss"); ax.grid(True, alpha=0.3)

ax = axes[1]
ee = [20*(i+1) for i in range(len(history["mAP"]))]
if history["mAP"]:
    ax.plot(ee, [m*100 for m in history["mAP"]], "r-o", label="mAP")
    ax.plot(ee, [r*100 for r in history["R1"]], "b-s", label="R1")
    if any(history["mAP_rr"]): ax.plot(ee, [m*100 for m in history["mAP_rr"]], "r--^", label="mAP(RR)")
ax.set_xlabel("Epoch"); ax.set_ylabel("%"); ax.set_title("Metrics"); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Export Models

In [ ]:
# -- Export TransReID --
tr_state = torch.load(OUTPUT_DIR / "transreid_veri_best.pth", map_location="cpu")
export_path = EXPORT_DIR / "vehicle_transreid_vit_base_veri776.pth"
torch.save({"state_dict": tr_state}, export_path)
print(f"Exported: {export_path} ({export_path.stat().st_size/1024**2:.1f} MB)")

# -- Metadata --
metadata = {
    "task": "vehicle_reid",
    "dataset": "veri776",
    "training": "TransReID ViT-Base CLIP (v8+, triplet, center_loss delayed@ep30, ε=0.15, 180ep)",
    "num_gpus": NUM_GPUS,
    "date": datetime.now().isoformat(),
    "model": {
        "architecture": VIT_MODEL,
        "type": "transreid",
        "tricks": ["SIE", "JPM", "BNNeck", "CE+LS(0.15)", "TripletLoss(m=0.3)",
                    "CenterLoss(delayed@ep30)", "CosLR", "RE", "CLIP-norm", "LLRD(0.75)",
                    "DataParallel"],
        "embedding_dim": 768,
        "input_size": [224, 224],
        "normalization": {"mean": CLIP_MEAN, "std": CLIP_STD},
        "best_mAP": float(best_mAP),
        "best_mAP_rr": float(history["mAP_rr"][-1]) if history["mAP_rr"] else None,
        "file": "vehicle_transreid_vit_base_veri776.pth",
        "training_hours": round(elapsed / 3600, 1),
        "epochs": EPOCHS,
        "backbone_lr": backbone_lr,
        "head_lr": head_lr,
        "llrd_factor": llrd_factor,
        "center_loss_weight": CENTER_WEIGHT,
        "center_loss_start_epoch": CENTER_START,
        "label_smoothing": 0.15,
        "metric_loss": "TripletLoss(margin=0.3)",
    },
    "total_gpu_hours": round(elapsed / 3600, 1),
    "v9_changes": [
        "Kept Triplet(0.3) from v8 (proven baseline)",
        "Label smoothing ε: 0.1 → 0.15: stronger regularisation for similar vehicles",
        "Epochs: 140 → 180: longer cosine tail for CLIP fine-tuning",
    ],
}

with open(EXPORT_DIR / "vehicle_reid_sota_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Total GPU time: {metadata['total_gpu_hours']}h")

In [ ]:
print("=" * 70)
print("VEHICLE ReID RESULTS — VeRi-776 (v8+)")
print("=" * 70)
print(f"\n{'Model':<30} {'mAP':>8} {'R1':>8} {'mAP(RR)':>10} {'R1(RR)':>10} {'Time':>8}")
print("-" * 76)

if history["mAP"]:
    rr = f"{history['mAP_rr'][-1]*100:.1f}%" if history["mAP_rr"] else "N/A"
    rr1 = f"{history['R1_rr'][-1]*100:.1f}%" if history["R1_rr"] else "N/A"
    print(f"{'TransReID ViT-Base (CLIP) v8+':<30} {best_mAP*100:>7.1f}% {history['R1'][-1]*100:>7.1f}% {rr:>10} {rr1:>10} {elapsed/3600:>7.1f}h")

print(f"\n{'SOTA Reference (CLIP-ReID)':<30} {'82.1%':>8} {'97.4%':>8} {'—':>10} {'—':>10}")
print(f"{'v8 Result (for comparison)':<30} {'80.4%':>8} {'96.3%':>8} {'83.1%':>10} {'97.4%':>10}")
print(f"\nExported to {EXPORT_DIR}/")
print("Place in project models/reid/ directory.")


## Local Integration

```yaml
# configs/default.yaml -- Stage 2 config
stage2:
  reid:
    model_name: transreid
    weights: models/reid/vehicle_transreid_vit_base_veri776.pth
    embedding_dim: 768
    input_size: [224, 224]
```

### v8+ Strategy
- **v6 proven base** (79.6% mAP): same LR, LLRD, augmentation
- **v8 proven Triplet** (80.4% mAP): kept Triplet(0.3) + center loss delayed@ep30
- **v8+ tweaks** (conservative, 2 changes only):
  1. **Label smoothing ε=0.15**: stronger regularisation for visually similar vehicles
  2. **180 epochs**: longer cosine tail extracts more from CLIP backbone
- **Parallel run** with 08b v9 (Circle Loss) for head-to-head comparison